# Create UCOP Awards (GRANT PATTERN, method-3 JSON-in-page-bundle)

University of California Office of the President awards from the publicly-published Research Grants Program Office (RGPO) database. UCOP administers research programs including TRDRP (Tobacco-Related Disease Research Program), CHRP (California HIV/AIDS Research Program), CBCRP (California Breast Cancer Research Program), and others.

**Prerequisites:**
- Run `scripts/local/ucop_to_s3.py` first to download and upload the data.

**Data source:** https://rgpogrants.ucop.edu/files/1614305/f480589/ucop.json (single JSON file behind the published HTML viewer at `/files/1614305/f480589/index.html`)
**S3 location:** `s3a://openalex-ingest/awards/ucop/ucop_awards.parquet`

**Awarding body:**
- funder_id: 4320333677
- display_name: "Office of the President, University of California"
- country: US
- ROR: none (UCOP has no ROR; verified via OpenAlex public API)
- DOI: 10.13039/100014576
- Crossref ID: 100014576
- alternate_titles: "UC Office of the President", "UCOP"

**Coverage (from 2026-05-22 local `--skip-upload` run):**
- 8,039 records, 1990-2026, total $2.37B
- 100% applicationid unique, 100% title/start/end-date, 99.8% approvedamount
- 98.7% grant_doi, 58.7% abstract, 65.6% priority
- Multi-PI rows present (some grants have 2-4 co-PIs); first PI = lead_investigator, second = co_lead_investigator

**Provenance:** `ucop_grants` (NOT an aggregator — direct from UCOP's own data feed, so no aggregator-suffix per runbook §2.1)

**Priority:** 106 (Vilcek at 105 is the current registry head)


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ucop_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/ucop/ucop_awards.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.ucop_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.ucop_raw;


## Step 1.5: Inspect Raw Data, Money Scan, and Native Key

UCOP exposes `approvedamount` as the amount column and is single-currency (USD, hardcoded — UCOP is a US funder and the source does not publish a currency field). Native award ID is `applicationid`, verified unique in the upload script. The runbook §1.5 RLIKE money/currency discovery scan is run anyway to confirm no unexpected money-shaped columns.

In [ ]:
%sql
-- Runbook §1.5 money-field discovery scan: every column whose name suggests money
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'ucop_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
-- Runbook §1.5 currency-field discovery scan (expected: no matches; USD is hardcoded)
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'ucop_raw'
  AND LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
-- Sample 5 raw rows to confirm shape before transformation
SELECT applicationid, title, approvedamount, startdate, enddate, program, awardtype, orgname, grant_doi
FROM openalex.awards.ucop_raw LIMIT 5;


In [ ]:
%sql
-- Amount distribution sanity check.
-- Expected from local validation: n=8024 non-null, min=$0, median=$110k, max=$11.7M, total $2.37B.
SELECT
    COUNT(*) AS total_rows,
    COUNT(approvedamount) AS rows_with_amount,
    SUM(CASE WHEN TRY_CAST(approvedamount AS DOUBLE) IS NOT NULL THEN 1 ELSE 0 END) AS rows_parseable_as_number,
    MIN(TRY_CAST(approvedamount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(approvedamount AS DOUBLE)) AS max_amount,
    ROUND(AVG(TRY_CAST(approvedamount AS DOUBLE)), 0) AS avg_amount,
    ROUND(SUM(TRY_CAST(approvedamount AS DOUBLE)) / 1e9, 2) AS total_usd_billions
FROM openalex.awards.ucop_raw;


In [ ]:
%sql
-- Native key uniqueness check (mandatory before building the awards table)
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT applicationid) AS distinct_application_ids,
    COUNT(DISTINCT LOWER(TRIM(applicationid))) AS distinct_lowercased
FROM openalex.awards.ucop_raw;


In [ ]:
%sql
-- Year distribution sanity (expected 1990-2026)
SELECT SUBSTR(startdate, 1, 4) AS start_year, COUNT(*) AS rows
FROM openalex.awards.ucop_raw
GROUP BY SUBSTR(startdate, 1, 4)
ORDER BY start_year;


In [ ]:
%sql
-- Contacts (PI) coverage
SELECT
    COUNT(*) AS total_rows,
    COUNT(contacts_json) AS rows_with_contacts,
    COUNT(CASE WHEN contacts_json IS NOT NULL AND contacts_json != '[]' THEN 1 END) AS rows_with_nonempty_contacts
FROM openalex.awards.ucop_raw;


In [ ]:
%sql
-- Look at one parsed contacts payload to confirm the from_json schema works
WITH parsed AS (
    SELECT
        applicationid,
        from_json(
            contacts_json,
            'ARRAY<STRUCT<institution:STRING, lastname:STRING, firstname:STRING, email:STRING, degree:STRING, role:STRING, subject:STRING>>'
        ) AS contacts
    FROM openalex.awards.ucop_raw
    WHERE contacts_json IS NOT NULL
    LIMIT 5
)
SELECT
    applicationid,
    size(contacts) AS pi_count,
    contacts[0].firstname AS first_firstname,
    contacts[0].lastname AS first_lastname,
    contacts[0].institution AS first_institution,
    contacts[0].role AS first_role
FROM parsed;


## Step 1.6: Fail-fast — Verify the UCOP Funder Row Exists

The Step 2 transform CROSS JOINs against `openalex.common.funder` to populate the `funder` struct. If F4320333677 is absent, the join silently emits zero rows and the Step 3 INSERT looks successful — the gap only surfaces downstream. **Runbook §2.2.4 mandatory.**

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320333677;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ucop_awards
USING delta
AS
WITH
ucop_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320333677
),

raw_with_pis AS (
    SELECT
        g.*,
        from_json(
            g.contacts_json,
            'ARRAY<STRUCT<institution:STRING, lastname:STRING, firstname:STRING, email:STRING, degree:STRING, role:STRING, subject:STRING>>'
        ) AS pis,
        TRY_CAST(REGEXP_REPLACE(g.approvedamount, '[^0-9.-]', '') AS DOUBLE) AS parsed_amount,
        COALESCE(
            TRY_TO_DATE(g.startdate, 'yyyy-MM-dd'),
            TRY_TO_DATE(SUBSTR(g.startdate, 1, 10), 'yyyy-MM-dd')
        ) AS parsed_start_date,
        COALESCE(
            TRY_TO_DATE(g.enddate, 'yyyy-MM-dd'),
            TRY_TO_DATE(SUBSTR(g.enddate, 1, 10), 'yyyy-MM-dd')
        ) AS parsed_end_date
    FROM openalex.awards.ucop_raw g
    WHERE g.applicationid IS NOT NULL
      AND TRIM(g.applicationid) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.applicationid)))) % 9000000000 as id,

        g.title as display_name,

        CASE
            WHEN g.abstract IS NOT NULL AND TRIM(g.abstract) != '' THEN g.abstract
            WHEN g.progressreportabbstract IS NOT NULL AND TRIM(g.progressreportabbstract) != '' THEN g.progressreportabbstract
            ELSE NULL
        END as description,

        f.funder_id,
        g.applicationid as funder_award_id,

        g.parsed_amount as amount,
        'USD' as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        CASE
            WHEN LOWER(COALESCE(g.awardtype, '')) RLIKE 'training|traineeship|fellowship|scholarship|postdoc|dissertation|predoc' THEN 'fellowship'
            ELSE 'grant'
        END as funding_type,

        COALESCE(NULLIF(TRIM(g.awardtype), ''), NULLIF(TRIM(g.program), '')) as funder_scheme,

        'ucop_grants' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        YEAR(g.parsed_start_date) as start_year,
        YEAR(g.parsed_end_date) as end_year,

        CASE
            WHEN g.pis IS NULL OR size(g.pis) = 0 THEN
                CAST(NULL AS STRUCT<
                    given_name:STRING,
                    family_name:STRING,
                    orcid:STRING,
                    role_start:DATE,
                    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
                >)
            ELSE struct(
                NULLIF(TRIM(try_element_at(g.pis, 1).firstname), '') as given_name,
                NULLIF(TRIM(try_element_at(g.pis, 1).lastname), '') as family_name,
                CAST(NULL AS STRING) as orcid,
                g.parsed_start_date as role_start,
                struct(
                    NULLIF(TRIM(try_element_at(g.pis, 1).institution), '') as name,
                    CAST(NULL AS STRING) as country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                ) as affiliation
            )
        END as lead_investigator,

        CASE
            WHEN g.pis IS NULL OR size(g.pis) < 2 THEN
                CAST(NULL AS STRUCT<
                    given_name:STRING,
                    family_name:STRING,
                    orcid:STRING,
                    role_start:DATE,
                    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
                >)
            ELSE struct(
                NULLIF(TRIM(try_element_at(g.pis, 2).firstname), '') as given_name,
                NULLIF(TRIM(try_element_at(g.pis, 2).lastname), '') as family_name,
                CAST(NULL AS STRING) as orcid,
                g.parsed_start_date as role_start,
                struct(
                    NULLIF(TRIM(try_element_at(g.pis, 2).institution), '') as name,
                    CAST(NULL AS STRING) as country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                ) as affiliation
            )
        END as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        CAST(NULL AS STRING) as landing_page_url,

        NULLIF(TRIM(g.grant_doi), '') as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.applicationid)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_with_pis g
    CROSS JOIN ucop_funder f
)

SELECT * FROM awards_transformed;


## Step 3: Insert Into `openalex_awards_raw` With Priority 106

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data (runbook §2.2.6).
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ucop_grants' AND priority = 106;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    106 as priority  -- UCOP priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.ucop_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.ucop_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.ucop_awards;


In [ ]:
%sql
-- §6.3 Data completeness. Expected: pct_title=100%, pct_amount~99.8%, pct_description~58.7%, pct_start_date=100%, pct_pi=100%.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator) as has_pi,
    COUNT(co_lead_investigator) as has_co_pi,
    COUNT(doi) as has_doi,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) as pct_start_date,
    ROUND(try_divide(COUNT(lead_investigator), COUNT(*)) * 100.0, 1) as pct_pi,
    ROUND(try_divide(COUNT(doi), COUNT(*)) * 100.0, 1) as pct_doi
FROM openalex.awards.ucop_awards;


In [ ]:
%sql
SELECT
    funder_award_id, display_name, amount, currency,
    start_date, end_date, funding_type, funder_scheme,
    lead_investigator.given_name AS pi_given,
    lead_investigator.family_name AS pi_family,
    lead_investigator.affiliation.name AS pi_affiliation
FROM openalex.awards.ucop_awards LIMIT 10;


In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.ucop_awards
GROUP BY funder.display_name;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.ucop_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 40;


In [ ]:
%sql
-- §6.7 Amount/currency fail-fast. Expected: pct_amount ~99.8%, distinct_currencies=1, max ~$11.7M.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_amount_nonzero,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(AVG(CASE WHEN amount > 0 THEN amount END), 0) AS avg_nonzero_amt,
    ROUND(SUM(amount) / 1e9, 2) AS total_usd_billions
FROM openalex.awards.ucop_awards;


In [ ]:
%sql
SELECT
    CASE
        WHEN lead_investigator IS NULL THEN 'no PI'
        WHEN co_lead_investigator IS NULL THEN 'single PI'
        ELSE 'multi PI'
    END AS pi_shape,
    COUNT(*) AS rows
FROM openalex.awards.ucop_awards
GROUP BY pi_shape
ORDER BY rows DESC;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.ucop_awards;
